# XGBoost NFL Running Back Performance Prediction

This notebook demonstrates how LLM-engineered features can enhance traditional ML models.

**Goal**: Predict weekly RB performance using:
- Statistical features (yards, touches, opponent rank)
- LLM-generated features (press ratings, injury likelihood, intuition grades)

In [42]:
%pip install nflreadpy anthropic python-dotenv pyarrow matplotlib xgboost scikit-learn

import nflreadpy as nflread
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import anthropic
import os
import pyarrow as pa

from dotenv import load_dotenv

load_dotenv()

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


True

## 1. Load NFL Running Back Data

In [21]:
# Load weekly player stats for 2025 season using nflreadpy
print("Loading 2025 NFL season data...")
weekly_stats_polars = nflread.load_player_stats([2025])

print(f"Data type: {type(weekly_stats_polars)}")
print(f"Shape: {weekly_stats_polars.shape}")

# Convert Polars to Pandas without pyarrow - use dict method
# This avoids the pyarrow dependency issue
weekly_stats = pd.DataFrame(weekly_stats_polars.to_dict(as_series=False))

# Filter for running backs only
rb_stats = weekly_stats[weekly_stats['position'] == 'RB'].copy()

# First, let's check what columns are available
print("\nAvailable columns:")
print([col for col in rb_stats.columns if 'team' in col.lower()])

# Select relevant statistical columns - using 'team' instead of 'recent_team'
stat_columns = [
    'player_id', 'player_name', 'week', 'season',
    'rushing_yards', 'rushing_tds', 'carries', 'targets', 'receptions',
    'receiving_yards', 'receiving_tds', 'fantasy_points_ppr',
    'opponent_team', 'team'
]

rb_stats = rb_stats[stat_columns].copy()

# Remove rows with missing rushing yards (our target)
rb_stats = rb_stats.dropna(subset=['rushing_yards'])

print(f"\nLoaded {len(rb_stats)} RB performances from 2025 season")
print(f"Unique players: {rb_stats['player_name'].nunique()}")
print(f"Weeks covered: {rb_stats['week'].min()} to {rb_stats['week'].max()}")

rb_stats.head()

Loading 2025 NFL season data...
Data type: <class 'polars.dataframe.frame.DataFrame'>
Shape: (4462, 114)

Available columns:
['team', 'opponent_team', 'special_teams_tds']

Loaded 375 RB performances from 2025 season
Unique players: 113
Weeks covered: 1 to 5


,player_id,player_name,week,season,rushing_yards,rushing_tds,carries,targets,receptions,receiving_yards,receiving_tds,fantasy_points_ppr,opponent_team,team
77,00-0032764,D.Henry,1,2025,169,2,18,1,1,13,0,29.2,BUF,BAL
96,00-0033280,C.McCaffrey,1,2025,69,0,22,10,9,73,0,23.2,SEA,SF
100,00-0033293,A.Jones,1,2025,23,0,8,3,3,44,1,15.7,CHI,MIN
108,00-0033526,S.Perine,1,2025,0,0,0,2,2,6,0,2.6,CLE,CIN
112,00-0033553,J.Conner,1,2025,39,0,12,4,4,5,1,14.4,NO,ARI


## 2. Engineer Basic Statistical Features

In [22]:
# Sort by player and week
rb_stats = rb_stats.sort_values(['player_id', 'week']).reset_index(drop=True)

# Create lagged features (previous week performance)
rb_stats['prev_rushing_yards'] = rb_stats.groupby('player_id')['rushing_yards'].shift(1)
rb_stats['prev_carries'] = rb_stats.groupby('player_id')['carries'].shift(1)
rb_stats['prev_fantasy_points'] = rb_stats.groupby('player_id')['fantasy_points_ppr'].shift(1)

# Rolling averages (last 3 weeks)
rb_stats['avg_rushing_yards_3w'] = rb_stats.groupby('player_id')['rushing_yards'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)
rb_stats['avg_carries_3w'] = rb_stats.groupby('player_id')['carries'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)

# Target variable: rushing yards this week
rb_stats['target_rushing_yards'] = rb_stats['rushing_yards']

# Filter for weeks 2-4 only
# - Week 1 dropped due to missing lagged features (no "previous week" data)
# - Week 5 excluded as games haven't occurred yet (Oct 4, 2025)
rb_stats = rb_stats[rb_stats['week'].isin([2, 3, 4])].copy()

print(f"After feature engineering: {len(rb_stats)} rows (weeks 2-4 only)")
print(f"Week 1: Dropped (no lagged features)")
print(f"Week 5: Excluded (games haven't happened yet)")
rb_stats.head()

After feature engineering: 274 rows (weeks 2-4 only)
Week 1: Dropped (no lagged features)
Week 5: Excluded (games haven't happened yet)


,player_id,player_name,week,season,rushing_yards,rushing_tds,carries,targets,receptions,receiving_yards,receiving_tds,fantasy_points_ppr,opponent_team,team,prev_rushing_yards,prev_carries,prev_fantasy_points,avg_rushing_yards_3w,avg_carries_3w,target_rushing_yards
0,00-0031687,R.Mostert,4,2025,62,0,4,1,1,11,0,8.3,CHI,LV,NaN,NaN,NaN,NaN,NaN,62
2,00-0032764,D.Henry,2,2025,23,0,11,0,0,0,0,2.3,CLE,BAL,169.0,18.0,29.2,169.000000,18.000000,23
3,00-0032764,D.Henry,3,2025,50,1,12,1,1,7,0,10.7,DET,BAL,23.0,11.0,2.3,96.000000,14.500000,50
4,00-0032764,D.Henry,4,2025,42,0,8,3,2,16,0,7.8,KC,BAL,50.0,12.0,10.7,80.666667,13.666667,42
6,00-0033280,C.McCaffrey,2,2025,55,0,13,7,6,52,1,22.7,NO,SF,69.0,22.0,23.2,69.000000,22.000000,55


## 3. LLM Feature Engineering

Now we'll use Claude to generate abstract features that capture qualitative information:
- **press_rating**: 1-10 rating based on recent news sentiment
- **injury_concern**: 1-5 scale of injury risk
- **intuition_grade**: 1-5 LLM assessment based on historical pattern recognition
- **opponent_defense_rating**: 1-10 rating of opponent run defense (higher = weaker defense)
- **oline_health**: 1-5 rating of offensive line health
- **vegas_sentiment**: 1-10 rating based on betting lines and expert picks

In [23]:
# Initialize Anthropic client
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

# We'll use a smaller sample for LLM feature generation (expensive operation)
# Focus on players with significant playing time
significant_rbs = rb_stats.groupby('player_id').agg({
    'carries': 'sum',
    'player_name': 'first'
}).reset_index()

significant_rbs = significant_rbs[significant_rbs['carries'] >= 10].sort_values('carries', ascending=False)
print(f"Focusing on {len(significant_rbs)} RBs with 10+ carries in 2025")
print(significant_rbs.head(10))

Focusing on 60 RBs with 10+ carries in 2025
     player_id  carries player_name
44  00-0037248       62      J.Cook
21  00-0035700       61    J.Jacobs
25  00-0036223       59    J.Taylor
12  00-0034844       59   S.Barkley
75  00-0039361       57    B.Irving
8   00-0033906       54    A.Kamara
58  00-0038542       52  B.Robinson
53  00-0037840       50  K.Williams
17  00-0035261       50   T.Pollard
90  00-0040122       49    A.Jeanty


In [24]:
import json
import os
from pathlib import Path
import re
import asyncio
from anthropic import AsyncAnthropic

# Create cache directory
CACHE_DIR = Path("llm_feature_cache")
CACHE_DIR.mkdir(exist_ok=True)

# Cache version - increment when prompts change
CACHE_VERSION = "v1"

# Initialize async client
async_client = AsyncAnthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))


def get_cache_key(feature_type, **kwargs):
    """Generate a unique cache key for a feature request"""
    # Sanitize values for filesystem safety
    sanitized_kwargs = {}
    for k, v in kwargs.items():
        # Remove special characters, keep only alphanumeric, spaces, hyphens
        sanitized_kwargs[k] = re.sub(r'[^\w\s-]', '', str(v))
    
    key_parts = [CACHE_VERSION, feature_type] + [f"{k}={v}" for k, v in sorted(sanitized_kwargs.items())]
    return "_".join(str(p).replace(" ", "_") for p in key_parts) + ".json"


def load_from_cache(feature_type, **kwargs):
    """Load a cached feature value if it exists"""
    cache_file = CACHE_DIR / get_cache_key(feature_type, **kwargs)
    if cache_file.exists():
        with open(cache_file, "r") as f:
            data = json.load(f)
            print(f"  [CACHE HIT] Loaded {feature_type} from cache")
            return data["value"]
    return None


def save_to_cache(feature_type, value, is_default=False, **kwargs):
    """Save a feature value to cache"""
    cache_file = CACHE_DIR / get_cache_key(feature_type, **kwargs)
    with open(cache_file, "w") as f:
        json.dump({"value": value, "kwargs": kwargs, "is_default": is_default}, f, indent=2)


async def get_llm_press_rating_async(player_name, week, year=2025):
    """Use Claude with web search to rate player press sentiment"""
    # Check cache first
    cached = load_from_cache(
        "press_rating", player_name=player_name, week=week, year=year
    )
    if cached is not None:
        return cached

    prompt = f"""
    Search for recent news and sentiment about NFL running back {player_name}
    around week {week} of the {year} season.

    Based on the press coverage, rate the player's public perception on a scale of 1-10:
    - 1-3: Negative coverage (injury concerns, poor performance, controversy)
    - 4-6: Neutral or mixed coverage
    - 7-10: Positive coverage (breakout performance, healthy, favorable matchup)

    First, briefly explain what you found in the search results (2-3 sentences).
    Then on a new line, provide ONLY a single number between 1 and 10.
    """

    try:
        response = await async_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}],
            tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 3}],
        )

        # Extract text from response
        text_content = ""
        for block in response.content:
            if block.type == "text":
                text_content += block.text

        # Try to extract the number from the last line
        lines = text_content.strip().split("\n")
        rating = None
        for line in reversed(lines):
            try:
                rating = int(line.strip())
                if 1 <= rating <= 10:
                    break
            except:
                continue
        result = rating if rating else 5
    except Exception as e:
        print(f"  Error in press_rating: {e}")
        result = 5  # Default neutral rating

    # Save to cache
    save_to_cache("press_rating", result, is_default=(result == 5), player_name=player_name, week=week, year=year)
    return result


async def get_llm_injury_concern_async(player_name, week, year=2025):
    """Use Claude with web search to assess injury likelihood"""
    # Check cache first
    cached = load_from_cache(
        "injury_concern", player_name=player_name, week=week, year=year
    )
    if cached is not None:
        return cached

    prompt = f"""
    Search for injury reports about NFL running back {player_name}
    around week {week} of the {year} season.

    Rate the injury concern level on a scale of 1-5:
    - 1: No injury concerns, fully healthy
    - 2: Minor issue, questionable but likely to play
    - 3: Moderate concern, may be limited
    - 4: Significant concern, doubtful to play
    - 5: Out or ruled out

    First, briefly explain what you found in injury reports (2-3 sentences).
    Then on a new line, provide ONLY a single number between 1 and 5.
    """

    try:
        response = await async_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}],
            tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 3}],
        )

        # Extract text from response
        text_content = ""
        for block in response.content:
            if block.type == "text":
                text_content += block.text

        lines = text_content.strip().split("\n")
        rating = None
        for line in reversed(lines):
            try:
                rating = int(line.strip())
                if 1 <= rating <= 5:
                    break
            except:
                continue
        result = rating if rating else 1
    except Exception as e:
        print(f"  Error in injury_concern: {e}")
        result = 1  # Default healthy

    # Save to cache
    save_to_cache(
        "injury_concern", result, is_default=(result == 1), player_name=player_name, week=week, year=year
    )
    return result


async def get_llm_intuition_grade_async(player_data_json):
    """Use Claude to provide an intuition-based grade on player's trajectory"""
    # Check cache first - use hash of player_data_json as key
    import hashlib

    data_hash = hashlib.md5(player_data_json.encode()).hexdigest()
    cached = load_from_cache("intuition_grade", data_hash=data_hash)
    if cached is not None:
        return cached

    prompt = f"""
    You are an expert NFL analyst. Review this running back's recent performance data:

    {player_data_json}

    Based on patterns, trends, and your expertise, give an intuition grade for their
    NEXT game performance on a scale of 1-5:
    - 1: Expect poor performance
    - 2: Below average expected
    - 3: Average expected
    - 4: Above average expected
    - 5: Breakout performance expected

    Consider workload trends, efficiency, recent game script, and momentum.
    Respond with ONLY a single number between 1 and 5.
    """

    try:
        response = await async_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=50,
            messages=[{"role": "user", "content": prompt}],
        )

        # Extract text from response
        text_content = ""
        for block in response.content:
            if block.type == "text":
                text_content += block.text

        rating = int(text_content.strip())
        result = max(1, min(5, rating))
    except Exception as e:
        print(f"  Error in intuition_grade: {e}")
        result = 3  # Default average

    # Save to cache
    save_to_cache("intuition_grade", result, is_default=(result == 3), data_hash=data_hash)
    return result


async def get_llm_opponent_defense_rating_async(opponent_team, week, year=2025):
    """Use Claude with web search to rate opponent run defense strength"""
    # Check cache first
    cached = load_from_cache(
        "opponent_defense", opponent_team=opponent_team, week=week, year=year
    )
    if cached is not None:
        return cached

    prompt = f"""
    Search for information about the {opponent_team} run defense 
    around week {week} of the {year} NFL season.

    Rate their run defense strength on a scale of 1-10:
    - 1-3: Elite run defense (top ranked, healthy, tough matchup for RBs)
    - 4-6: Average run defense
    - 7-10: Weak run defense (injuries, poor ranking, favorable for RBs)

    First, briefly explain what you found about their run defense (2-3 sentences).
    Then on a new line, provide ONLY a single number between 1 and 10.
    """

    try:
        response = await async_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}],
            tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 3}],
        )

        text_content = ""
        for block in response.content:
            if block.type == "text":
                text_content += block.text

        lines = text_content.strip().split("\n")
        rating = None
        for line in reversed(lines):
            try:
                rating = int(line.strip())
                if 1 <= rating <= 10:
                    break
            except:
                continue
        result = rating if rating else 5
    except Exception as e:
        print(f"  Error in opponent_defense: {e}")
        result = 5  # Default average

    # Save to cache
    save_to_cache(
        "opponent_defense", result, is_default=(result == 5), opponent_team=opponent_team, week=week, year=year
    )
    return result


async def get_llm_oline_health_async(team, week, year=2025):
    """Use Claude with web search to assess offensive line health"""
    # Check cache first
    cached = load_from_cache("oline_health", team=team, week=week, year=year)
    if cached is not None:
        return cached

    prompt = f"""
    Search for offensive line injury reports for the {team} 
    around week {week} of the {year} NFL season.

    Rate the offensive line health on a scale of 1-5:
    - 1: Multiple starters out or questionable, severe injuries
    - 2: One starter out, or multiple backups playing
    - 3: Minor injuries, some game-time decisions
    - 4: Mostly healthy, minor issues only
    - 5: Fully healthy, all starters playing

    First, briefly explain what you found in injury reports (2-3 sentences).
    Then on a new line, provide ONLY a single number between 1 and 5.
    """

    try:
        response = await async_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}],
            tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 3}],
        )

        text_content = ""
        for block in response.content:
            if block.type == "text":
                text_content += block.text

        lines = text_content.strip().split("\n")
        rating = None
        for line in reversed(lines):
            try:
                rating = int(line.strip())
                if 1 <= rating <= 5:
                    break
            except:
                continue
        result = rating if rating else 4
    except Exception as e:
        print(f"  Error in oline_health: {e}")
        result = 4  # Default mostly healthy

    # Save to cache
    save_to_cache("oline_health", result, is_default=(result == 4), team=team, week=week, year=year)
    return result


async def get_llm_vegas_sentiment_async(player_name, week, year=2025):
    """Use Claude with web search to assess Vegas and expert sentiment"""
    # Check cache first
    cached = load_from_cache(
        "vegas_sentiment", player_name=player_name, week=week, year=year
    )
    if cached is not None:
        return cached

    prompt = f"""
    Search for betting lines, prop lines, and expert picks for NFL running back {player_name}
    for week {week} of the {year} season.

    Rate the Vegas/expert sentiment on a scale of 1-10:
    - 1-3: Bearish - low prop lines, experts fading, unfavorable odds
    - 4-6: Neutral - average expectations
    - 7-10: Bullish - high prop lines, experts hyping, favorable odds

    First, briefly explain what you found about betting lines and expert picks (2-3 sentences).
    Then on a new line, provide ONLY a single number between 1 and 10.
    """

    try:
        response = await async_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}],
            tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 3}],
        )

        text_content = ""
        for block in response.content:
            if block.type == "text":
                text_content += block.text

        lines = text_content.strip().split("\n")
        rating = None
        for line in reversed(lines):
            try:
                rating = int(line.strip())
                if 1 <= rating <= 10:
                    break
            except:
                continue
        result = rating if rating else 5
    except Exception as e:
        print(f"  Error in vegas_sentiment: {e}")
        result = 5  # Default neutral

    # Save to cache
    save_to_cache(
        "vegas_sentiment", result, is_default=(result == 5), player_name=player_name, week=week, year=year
    )
    return result


async def get_all_llm_features_for_row(row):
    """Get all 6 LLM features for a single row in parallel"""
    player_name = row['player_name']
    week = row['week']
    year = row['season']
    opponent = row['opponent_team']
    team = row['team']
    
    # Get historical data for intuition grade
    player_history = rb_stats[
        (rb_stats['player_id'] == row['player_id']) & 
        (rb_stats['week'] < week)
    ][['week', 'rushing_yards', 'carries', 'fantasy_points_ppr']].tail(4)
    
    player_data_json = player_history.to_json(orient='records')
    
    # Run all 6 feature calls in parallel
    results = await asyncio.gather(
        get_llm_press_rating_async(player_name, week, year),
        get_llm_injury_concern_async(player_name, week, year),
        get_llm_intuition_grade_async(player_data_json),
        get_llm_opponent_defense_rating_async(opponent, week, year),
        get_llm_oline_health_async(team, week, year),
        get_llm_vegas_sentiment_async(player_name, week, year),
        return_exceptions=True  # Don't fail entire batch if one fails
    )
    
    # Handle any exceptions
    default_values = [5, 1, 3, 5, 4, 5]
    processed_results = []
    for i, result in enumerate(results):
        if isinstance(result, Exception):
            print(f"  Error in feature {i}: {result}")
            processed_results.append(default_values[i])
        else:
            processed_results.append(result)
    
    return processed_results

In [25]:
# Generate LLM features for our significant RBs
# Using async/parallel processing with max 5 concurrent API calls

# Initialize columns
rb_stats['press_rating'] = np.nan
rb_stats['injury_concern'] = np.nan
rb_stats['intuition_grade'] = np.nan
rb_stats['opponent_defense_rating'] = np.nan
rb_stats['oline_health'] = np.nan
rb_stats['vegas_sentiment'] = np.nan

# Filter for weeks 2-4 only (week 5 hasn't happened yet as of Oct 4, 2025)
# Also filter for significant RBs (10+ carries)
top_players = significant_rbs['player_id'].tolist()

rows_to_process = rb_stats[
    (rb_stats['player_id'].isin(top_players)) & 
    (rb_stats['week'].isin([2, 3, 4]))  # Only weeks 2-4
].copy()

print(f"Generating LLM features for {len(rows_to_process)} player-week samples")
print(f"Weeks: 2-4 (week 5 excluded as games haven't occurred yet)")
print(f"Players: {len(top_players)} RBs with 10+ carries")
print(f"Processing with max 5 concurrent API calls to respect rate limits\\n")


async def process_all_rows_with_limit(rows_df, max_concurrent=5):
    """Process all rows with a limit on concurrent API calls"""
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def process_with_semaphore(idx, row):
        async with semaphore:
            player_name = row['player_name']
            week = row['week']
            print(f"Processing {player_name} - Week {week}")
            results = await get_all_llm_features_for_row(row)
            return idx, results
    
    # Create tasks for all rows
    tasks = [
        process_with_semaphore(idx, row) 
        for idx, row in rows_df.iterrows()
    ]
    
    # Run all tasks with semaphore limiting concurrency
    results = await asyncio.gather(*tasks)
    return results


# Run the async processing
import nest_asyncio
nest_asyncio.apply()  # Allow nested event loops in Jupyter

results = await process_all_rows_with_limit(rows_to_process, max_concurrent=5)

# Update the dataframe with results
feature_names = ['press_rating', 'injury_concern', 'intuition_grade', 
                 'opponent_defense_rating', 'oline_health', 'vegas_sentiment']

for idx, feature_values in results:
    for feature_name, value in zip(feature_names, feature_values):
        rb_stats.at[idx, feature_name] = value

print(f"\\nLLM feature generation complete!")
print(f"Processed {len(results)} player-week samples")
print(f"Features generated: {feature_names}")

Generating LLM features for 173 player-week samples
Weeks: 2-4 (week 5 excluded as games haven't occurred yet)
Players: 60 RBs with 10+ carries
Processing with max 5 concurrent API calls to respect rate limits\n
Processing D.Henry - Week 2
Processing D.Henry - Week 3
Processing D.Henry - Week 4
Processing C.McCaffrey - Week 2
Processing C.McCaffrey - Week 3
Processing C.McCaffrey - Week 4
Processing J.Conner - Week 2
  [CACHE HIT] Loaded intuition_grade from cache
Processing J.Conner - Week 3
Processing A.Kamara - Week 2
  [CACHE HIT] Loaded intuition_grade from cache
Processing A.Kamara - Week 3
Processing A.Kamara - Week 4
Processing K.Hunt - Week 2
  [CACHE HIT] Loaded intuition_grade from cache
Processing K.Hunt - Week 3
Processing K.Hunt - Week 4
Processing N.Chubb - Week 2
  [CACHE HIT] Loaded intuition_grade from cache
Processing N.Chubb - Week 3
Processing N.Chubb - Week 4
Processing S.Barkley - Week 2
  [CACHE HIT] Loaded intuition_grade from cache
Processing S.Barkley - Week 

In [32]:
# Check the data with new features
rb_stats_with_llm = rb_stats[rb_stats['press_rating'].notna()].copy()
print(f"Samples with LLM features: {len(rb_stats_with_llm)}")
rb_stats_with_llm[[
    'player_name', 'week', 'rushing_yards', 
    'press_rating', 'injury_concern', 'intuition_grade',
    'opponent_defense_rating', 'oline_health', 'vegas_sentiment'
]].head(10)

Samples with LLM features: 173


,player_name,week,rushing_yards,press_rating,injury_concern,intuition_grade,opponent_defense_rating,oline_health,vegas_sentiment
2,D.Henry,2,23,5.0,1.0,3.0,2.0,4.0,5.0
3,D.Henry,3,50,5.0,1.0,2.0,2.0,4.0,7.0
4,D.Henry,4,42,5.0,1.0,3.0,5.0,4.0,5.0
6,C.McCaffrey,2,55,5.0,1.0,3.0,8.0,3.0,5.0
7,C.McCaffrey,3,52,5.0,1.0,3.0,2.0,4.0,5.0
8,C.McCaffrey,4,49,5.0,1.0,3.0,5.0,3.0,5.0
17,J.Conner,2,34,5.0,1.0,3.0,8.0,4.0,5.0
18,J.Conner,3,22,2.0,1.0,2.0,5.0,3.0,4.0
25,A.Kamara,2,99,3.0,1.0,3.0,3.0,3.0,6.0
26,A.Kamara,3,42,5.0,1.0,3.0,5.0,4.0,5.0


## 4. Train XGBoost Model

In [33]:
# Prepare dataset with LLM features
model_data = rb_stats_with_llm.copy()

# Statistical features
stat_features = [
    'prev_rushing_yards', 'prev_carries', 'prev_fantasy_points',
    'avg_rushing_yards_3w', 'avg_carries_3w'
]

# LLM features
llm_features = [
    'press_rating', 'injury_concern', 'intuition_grade',
    'opponent_defense_rating', 'oline_health', 'vegas_sentiment'
]

all_features = stat_features + llm_features
target = 'target_rushing_yards'

# Remove rows with missing values
model_data = model_data[all_features + [target]].dropna()

print(f"Training samples: {len(model_data)}")
print(f"Statistical features: {stat_features}")
print(f"LLM features: {llm_features}")
print(f"Target: {target}")

Training samples: 170
Statistical features: ['prev_rushing_yards', 'prev_carries', 'prev_fantasy_points', 'avg_rushing_yards_3w', 'avg_carries_3w']
LLM features: ['press_rating', 'injury_concern', 'intuition_grade', 'opponent_defense_rating', 'oline_health', 'vegas_sentiment']
Target: target_rushing_yards


In [34]:
# Train/test split (chronological - use earlier weeks for training)
# For 2025 season with limited weeks, we'll use weeks 2-3 for training and week 4 for testing
model_data_with_week = rb_stats_with_llm.copy()

# Check what weeks we have
print(f"Available weeks: {sorted(model_data_with_week['week'].unique())}")
print(f"Week counts:\n{model_data_with_week['week'].value_counts().sort_index()}")

# Use explicit week split: train on weeks 2-3, test on week 4
# This is more robust for early season data
train_weeks = [2, 3]
test_weeks = [4]

train_indices = model_data_with_week[model_data_with_week["week"].isin(train_weeks)].index
test_indices = model_data_with_week[model_data_with_week["week"].isin(test_weeks)].index

# Now filter model_data to only include features + target
model_data = model_data_with_week[all_features + [target]].dropna()

# Split based on indices that exist in model_data
train_data = model_data.loc[model_data.index.intersection(train_indices)]
test_data = model_data.loc[model_data.index.intersection(test_indices)]

X_train = train_data[all_features]
y_train = train_data[target]
X_test = test_data[all_features]
y_test = test_data[target]

print(f"\nTrain set (weeks {train_weeks}): {len(X_train)} samples")
print(f"Test set (weeks {test_weeks}): {len(X_test)} samples")

# Warn if test set is too small
if len(X_test) < 20:
    print(f"\n⚠️  WARNING: Test set is small ({len(X_test)} samples). Results may be less reliable.")
if len(X_train) < 20:
    print(f"\n⚠️  WARNING: Train set is small ({len(X_train)} samples). Model may underfit.")

Available weeks: [2, 3, 4]
Week counts:
2    58
3    59
4    56
Name: week, dtype: int64

Train set (weeks [2, 3]): 114 samples
Test set (weeks [4]): 56 samples


In [35]:
# Train XGBoost model
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

# Make predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Evaluate
print("\n=== Model Performance ===")
print(f"\nTrain Set:")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.2f}")
print(f"  MAE: {mean_absolute_error(y_train, y_pred_train):.2f}")
print(f"  R²: {r2_score(y_train, y_pred_train):.3f}")

print(f"\nTest Set:")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.2f}")
print(f"  MAE: {mean_absolute_error(y_test, y_pred_test):.2f}")
print(f"  R²: {r2_score(y_test, y_pred_test):.3f}")


=== Model Performance ===

Train Set:
  RMSE: 2.59
  MAE: 1.78
  R²: 0.994

Test Set:
  RMSE: 26.03
  MAE: 20.61
  R²: 0.393


## 5. Analyze Feature Importance

In [36]:
# Get feature importance
importance_df = pd.DataFrame({
    'feature': all_features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

# Categorize features
importance_df['type'] = importance_df['feature'].apply(
    lambda x: 'LLM' if x in llm_features else 'Statistical'
)

print("\n=== Feature Importance ===")
print(importance_df.to_string(index=False))

# Calculate aggregate importance by type
print("\n=== Importance by Feature Type ===")
importance_by_type = importance_df.groupby('type')['importance'].sum()
print(importance_by_type)
print(f"\nLLM features contribution: {importance_by_type.get('LLM', 0):.1%}")
print(f"Statistical features contribution: {importance_by_type.get('Statistical', 0):.1%}")


=== Feature Importance ===
                feature  importance        type
         avg_carries_3w    0.379914 Statistical
           press_rating    0.144211         LLM
        intuition_grade    0.088397         LLM
   avg_rushing_yards_3w    0.076415 Statistical
    prev_fantasy_points    0.060189 Statistical
opponent_defense_rating    0.059846         LLM
           oline_health    0.046561         LLM
           prev_carries    0.044315 Statistical
     prev_rushing_yards    0.041870 Statistical
        vegas_sentiment    0.030895         LLM
         injury_concern    0.027388         LLM

=== Importance by Feature Type ===
type
LLM            0.397297
Statistical    0.602703
Name: importance, dtype: float32

LLM features contribution: 39.7%
Statistical features contribution: 60.3%


In [ ]:
import plotly.graph_objects as go

# Create horizontal bar chart
colors = ['#FF6B6B' if t == 'LLM' else '#4ECDC4' for t in importance_df['type']]

fig = go.Figure(data=[
    go.Bar(
        y=importance_df['feature'],
        x=importance_df['importance'],
        orientation='h',
        marker=dict(color=colors),
        text=importance_df['importance'].round(3),
        textposition='auto',
    )
])

fig.update_layout(
    title='XGBoost Feature Importance: LLM vs Statistical Features',
    xaxis_title='Importance',
    yaxis_title='Feature',
    height=500,
    showlegend=False
)

fig.show()

## 6. Compare: Model With vs Without LLM Features

In [ ]:
# Train baseline model without LLM features
model_baseline = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

X_train_baseline = X_train[stat_features]
X_test_baseline = X_test[stat_features]

model_baseline.fit(X_train_baseline, y_train)
y_pred_baseline = model_baseline.predict(X_test_baseline)

# Compare performance
print("\n=== Model Comparison ===")
print(f"\nBaseline (Statistical Features Only):")
print(f"  Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_baseline)):.2f}")
print(f"  Test MAE: {mean_absolute_error(y_test, y_pred_baseline):.2f}")
print(f"  Test R²: {r2_score(y_test, y_pred_baseline):.3f}")

print(f"\nEnhanced (Statistical + LLM Features):")
print(f"  Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.2f}")
print(f"  Test MAE: {mean_absolute_error(y_test, y_pred_test):.2f}")
print(f"  Test R²: {r2_score(y_test, y_pred_test):.3f}")

rmse_improvement = (np.sqrt(mean_squared_error(y_test, y_pred_baseline)) - 
                    np.sqrt(mean_squared_error(y_test, y_pred_test)))
print(f"\nRMSE Improvement: {rmse_improvement:.2f} yards")

In [53]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Calculate metrics for both models
rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
rmse_enhanced = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae_baseline = mean_absolute_error(y_test, y_pred_baseline)
mae_enhanced = mean_absolute_error(y_test, y_pred_test)
r2_baseline = r2_score(y_test, y_pred_baseline)
r2_enhanced = r2_score(y_test, y_pred_test)

# Create subplots: 1 row, 2 columns
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Model Performance Metrics', 'Prediction Accuracy Scatter'),
    specs=[[{"type": "bar"}, {"type": "scatter"}]]
)

# Subplot 1: Bar chart comparing metrics
# Add annotations to indicate direction
metrics = ['RMSE<br>(lower is better)', 'MAE<br>(lower is better)', 'R²<br>(higher is better)']
baseline_values = [rmse_baseline, mae_baseline, r2_baseline]
enhanced_values = [rmse_enhanced, mae_enhanced, r2_enhanced]

fig.add_trace(
    go.Bar(name='Baseline (Stats Only)', x=metrics, y=baseline_values, 
           marker_color='#4ECDC4', text=[f'{v:.2f}' for v in baseline_values],
           textposition='outside'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(name='Enhanced (Stats + LLM)', x=metrics, y=enhanced_values,
           marker_color='#FF6B6B', text=[f'{v:.2f}' for v in enhanced_values],
           textposition='outside'),
    row=1, col=1
)

# Subplot 2: Scatter plot of actual vs predicted
fig.add_trace(
    go.Scatter(x=y_test, y=y_pred_baseline, mode='markers',
               name='Baseline', marker=dict(color='#4ECDC4', size=8, opacity=0.6)),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(x=y_test, y=y_pred_test, mode='markers',
               name='Enhanced', marker=dict(color='#FF6B6B', size=8, opacity=0.6)),
    row=1, col=2
)

# Add perfect prediction line
max_yards = max(y_test.max(), y_pred_baseline.max(), y_pred_test.max())
fig.add_trace(
    go.Scatter(x=[0, max_yards], y=[0, max_yards], mode='lines',
               name='Perfect Prediction', line=dict(color='gray', dash='dash')),
    row=1, col=2
)

# Update layout
fig.update_xaxes(title_text="Metric", row=1, col=1)
fig.update_yaxes(title_text="Value", row=1, col=1)
fig.update_xaxes(title_text="Actual Rushing Yards", row=1, col=2)
fig.update_yaxes(title_text="Predicted Rushing Yards", row=1, col=2)

fig.update_layout(
    title_text="Impact of LLM Feature Engineering on Prediction Accuracy",
    showlegend=True,
    height=500,
    width=1200
)

fig.show()

# Print improvement summary
print("\n=== Improvement Summary ===")
print(f"RMSE Improvement: {rmse_baseline - rmse_enhanced:.2f} yards ({(rmse_baseline - rmse_enhanced)/rmse_baseline*100:.1f}%) ⬇️")
print(f"MAE Improvement: {mae_baseline - mae_enhanced:.2f} yards ({(mae_baseline - mae_enhanced)/mae_baseline*100:.1f}%) ⬇️")
print(f"R² Improvement: {r2_enhanced - r2_baseline:.3f} ({(r2_enhanced - r2_baseline)/abs(r2_baseline)*100:.1f}%) ⬆️")


=== Improvement Summary ===
RMSE Improvement: 7.80 yards (23.0%) ⬇️
MAE Improvement: 4.37 yards (17.5%) ⬇️
R² Improvement: 0.418 (1651.8%) ⬆️


## Week 5 Predictions vs Actual Results (Lions vs 49ers - TNF Oct 2)

In [55]:

# Get all week 5 data with actual results
week5_all = rb_stats[rb_stats['week'] == 5].copy()

print(f"Total Week 5 samples in dataset: {len(week5_all)}")
print(f"Week 5 samples with LLM features: {len(rb_stats_with_llm[rb_stats_with_llm['week'] == 5])}")

# Get week 5 data that has both LLM features AND actual results
week5_with_features = rb_stats_with_llm[rb_stats_with_llm['week'] == 5].copy()

if len(week5_with_features) > 0:
    print(f"\n✅ Found {len(week5_with_features)} RBs from Week 5 TNF (Lions vs 49ers) with complete data")
    
    # Prepare features
    X_week5 = week5_with_features[all_features].dropna()
    X_week5_baseline = X_week5[stat_features]
    
    # Make predictions
    week5_pred_baseline = model_baseline.predict(X_week5_baseline)
    week5_pred_enhanced = model.predict(X_week5)
    
    # Get actual results
    y_week5_actual = week5_with_features.loc[X_week5.index, 'rushing_yards'].values
    
    # Calculate errors
    baseline_errors = np.abs(week5_pred_baseline - y_week5_actual)
    enhanced_errors = np.abs(week5_pred_enhanced - y_week5_actual)
    
    # Create results dataframe
    results_df = pd.DataFrame({
        'Player': week5_with_features.loc[X_week5.index, 'player_name'].values,
        'Team': week5_with_features.loc[X_week5.index, 'team'].values,
        'Actual Yards': y_week5_actual,
        'Baseline Pred': week5_pred_baseline.round(1),
        'Baseline Error': baseline_errors.round(1),
        'LLM Pred': week5_pred_enhanced.round(1),
        'LLM Error': enhanced_errors.round(1),
        'Error Reduction': (baseline_errors - enhanced_errors).round(1)
    })
    
    # Sort by actual yards
    results_df = results_df.sort_values('Actual Yards', ascending=False)
    
    print("\n" + "="*90)
    print("🏈 WEEK 5 TNF: LIONS @ 49ERS - MODEL PERFORMANCE 🏈")
    print("="*90)
    print("\nPredictions vs Actual Results:\n")
    print(results_df.to_string(index=False))
    
    # Calculate aggregate metrics
    baseline_mae = baseline_errors.mean()
    enhanced_mae = enhanced_errors.mean()
    improvement = ((baseline_mae - enhanced_mae) / baseline_mae) * 100
    
    print("\n" + "="*90)
    print("📊 MODEL PERFORMANCE SUMMARY")
    print("="*90)
    print(f"Baseline Model MAE: {baseline_mae:.2f} yards")
    print(f"LLM Enhanced Model MAE: {enhanced_mae:.2f} yards")
    print(f"Improvement: {improvement:.1f}% {'📈' if improvement > 0 else '📉'}")
    print("="*90)
    
    # Tweet format
    print("\n\n📱 TWEET-READY FORMAT:\n")
    print("🏈 Week 5 TNF Predictions vs Reality")
    print("Lions @ 49ers\n")
    for idx, row in results_df.iterrows():
        better_model = "LLM✅" if row['LLM Error'] < row['Baseline Error'] else "Base✅" if row['Baseline Error'] < row['LLM Error'] else "Tie"
        print(f"{row['Player']} ({row['Team']}): {row['Actual Yards']:.0f} yds")
        print(f"  Base: {row['Baseline Pred']:.0f} (off by {row['Baseline Error']:.0f}) | LLM: {row['LLM Pred']:.0f} (off by {row['LLM Error']:.0f}) {better_model}")
    print(f"\nLLM model was {improvement:.0f}% more accurate overall!")
    
else:
    print("\n⚠️ No Week 5 data with LLM features found.")
    print("This likely means LLM features weren't generated for Week 5 games yet.")

Total Week 5 samples in dataset: 0
Week 5 samples with LLM features: 0

⚠️ No Week 5 data with LLM features found.
This likely means LLM features weren't generated for Week 5 games yet.


In [54]:
## Week 5 Predictions - Ready for X/Twitter Post

# Get week 5 data for players with LLM features
week5_data = rb_stats_with_llm[rb_stats_with_llm['week'] == 5].copy()

# If no week 5 data with LLM features, we need to generate predictions for top players
# Let's use the players from our training set and create week 5 features
print(f"Week 5 samples with LLM features: {len(week5_data)}")

if len(week5_data) == 0:
    print("\nNo week 5 data found. Using most recent data from top performers...")
    # Get top 10 players by average performance
    top_performers = rb_stats_with_llm.groupby('player_name').agg({
        'rushing_yards': 'mean',
        'player_id': 'first',
        'carries': 'mean'
    }).sort_values('rushing_yards', ascending=False).head(10)
    
    print(f"\nTop 10 RBs by average rushing yards:")
    print(top_performers)
else:
    # Use actual week 5 data
    print(f"\nMaking predictions for {len(week5_data)} RBs in week 5...")
    
    # Prepare features
    X_week5 = week5_data[all_features].dropna()
    X_week5_baseline = X_week5[stat_features]
    
    # Make predictions
    week5_pred_baseline = model_baseline.predict(X_week5_baseline)
    week5_pred_enhanced = model.predict(X_week5)
    
    # Create results dataframe
    predictions_df = pd.DataFrame({
        'Player': week5_data.loc[X_week5.index, 'player_name'].values,
        'Baseline Pred': week5_pred_baseline,
        'LLM Enhanced Pred': week5_pred_enhanced,
        'Difference': week5_pred_enhanced - week5_pred_baseline,
        'Avg Last 3 Weeks': week5_data.loc[X_week5.index, 'avg_rushing_yards_3w'].values
    })
    
    # Sort by LLM Enhanced prediction
    predictions_df = predictions_df.sort_values('LLM Enhanced Pred', ascending=False)
    
    # Round for display
    predictions_df['Baseline Pred'] = predictions_df['Baseline Pred'].round(1)
    predictions_df['LLM Enhanced Pred'] = predictions_df['LLM Enhanced Pred'].round(1)
    predictions_df['Difference'] = predictions_df['Difference'].round(1)
    predictions_df['Avg Last 3 Weeks'] = predictions_df['Avg Last 3 Weeks'].round(1)
    
    print("\n" + "="*80)
    print("🏈 NFL RB WEEK 5 RUSHING YARDS PREDICTIONS 🏈")
    print("="*80)
    print("\nComparing Baseline (Stats Only) vs LLM-Enhanced Feature Engineering\n")
    print(predictions_df.to_string(index=False))
    print("\n" + "="*80)
    print(f"Model Performance: LLM features improved prediction accuracy by {(rmse_baseline - rmse_enhanced)/rmse_baseline*100:.1f}%")
    print("="*80)
    
    # Create a tweet-friendly version (top 5)
    print("\n\n📱 TWEET-READY FORMAT (Top 5 Predictions):\n")
    print("🏈 Week 5 RB Predictions: LLM-Enhanced ML Model\n")
    for idx, row in predictions_df.head(5).iterrows():
        direction = "📈" if row['Difference'] > 0 else "📉" if row['Difference'] < 0 else "➡️"
        print(f"{row['Player']}: {row['LLM Enhanced Pred']:.0f} yds {direction}")
    print(f"\nLLM feature engineering improved prediction accuracy by {(rmse_baseline - rmse_enhanced)/rmse_baseline*100:.0f}%")

Week 5 samples with LLM features: 0

No week 5 data found. Using most recent data from top performers...

Top 10 RBs by average rushing yards:
             rushing_yards   player_id    carries
player_name                                      
J.Cook          119.000000  00-0037248  20.666667
J.Taylor        114.333333  00-0036223  19.666667
J.Dobbins        86.666667  00-0036158  13.666667
J.Williams       86.000000  00-0036997  16.000000
J.Gibbs          84.000000  00-0039139  16.333333
T.Etienne        83.666667  00-0036973  16.333333
A.Jeanty         81.333333  00-0040122  16.333333
K.Williams       79.000000  00-0037840  16.666667
Q.Judkins        79.000000  00-0040784  16.333333
K.Walker         74.666667  00-0038134  16.000000
